In [1]:
"""
FEATURE ENGINEERING
==========================
Notebook: 02_Feature_Engineering.ipynb
Date: [Dec 09 2025]

Goal: Create domain-specific features for bridge condition prediction
Input: X_train_final.csv, X_test_final.csv, y_train.csv, y_test.csv
Output: X_train_engineered.csv, X_test_engineered.csv
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("DAY 1: FEATURE ENGINEERING")
print("="*70)

# Load preprocessed data
print("\nLoading preprocessed data...")
X_train = pd.read_csv('X_train_final_v2.csv')
X_test = pd.read_csv('X_test_final_v2.csv')
y_train = pd.read_csv('y_train.csv').squeeze()
y_test = pd.read_csv('y_test.csv').squeeze()

print(f"Training data: {X_train.shape}")
print(f"Test data: {X_test.shape}")
print(f"\nOriginal features: {X_train.shape[1]}")

# Store original column names
original_columns = X_train.columns.tolist()

DAY 1: FEATURE ENGINEERING

Loading preprocessed data...
Training data: (1236125, 102)
Test data: (624193, 102)

Original features: 102


In [2]:
"""
Helper function to create features on both train and test consistently
"""

def create_engineered_features(df, is_train=True):
    """
    Create engineered features WITHOUT using condition ratings
    """

    df_eng = df.copy()
    features_created = []
    current_year = 2025

    # ============================================================
    # 1. AGE FEATURES (SAME AS BEFORE - VALID)
    # ============================================================
    if 'YEAR_BUILT_027' in df.columns:
        df_eng['bridge_age'] = current_year - df['YEAR_BUILT_027']
        df_eng['age_category'] = pd.cut(
            df_eng['bridge_age'],
            bins=[-np.inf, 10, 30, 50, np.inf],
            labels=[0, 1, 2, 3]
        ).astype(float)
        df_eng['is_very_old'] = (df_eng['bridge_age'] > 50).astype(int)
        features_created.extend(['bridge_age', 'age_category', 'is_very_old'])

    if 'YEAR_RECONSTRUCTED_106' in df.columns:
        df_eng['years_since_reconstruction'] = current_year - df['YEAR_RECONSTRUCTED_106']
        df_eng['years_since_reconstruction'] = df_eng['years_since_reconstruction'].fillna(999)
        df_eng['was_reconstructed'] = (df['YEAR_RECONSTRUCTED_106'].notna()).astype(int)
        features_created.extend(['years_since_reconstruction', 'was_reconstructed'])

    # ============================================================
    # 2. TRAFFIC FEATURES
    # ============================================================
    if 'ADT_029' in df.columns:
        df_eng['log_adt'] = np.log1p(df['ADT_029'])
        features_created.append('log_adt')

        if is_train:
            traffic_90 = df['ADT_029'].quantile(0.9)
        else:
            traffic_90 = X_train['ADT_029'].quantile(0.9)

        df_eng['is_high_traffic'] = (df['ADT_029'] > traffic_90).astype(int)
        features_created.append('is_high_traffic')

        if 'TRAFFIC_LANES_ON_028A' in df.columns:
            df_eng['traffic_per_lane'] = df['ADT_029'] / (df['TRAFFIC_LANES_ON_028A'] + 1)
            features_created.append('traffic_per_lane')

        if 'PERCENT_ADT_TRUCK_109' in df.columns:
            df_eng['truck_impact'] = df['PERCENT_ADT_TRUCK_109'] * df['ADT_029'] / 100
            df_eng['is_heavy_truck'] = (df['PERCENT_ADT_TRUCK_109'] > 10).astype(int)
            features_created.extend(['truck_impact', 'is_heavy_truck'])

    # ============================================================
    # 3. STRUCTURAL FEATURES
    # ============================================================
    if 'DECK_WIDTH_MT_052' in df.columns and 'STRUCTURE_LEN_MT_049' in df.columns:
        df_eng['deck_to_structure_ratio'] = df['DECK_WIDTH_MT_052'] / (df['STRUCTURE_LEN_MT_049'] + 1)
        features_created.append('deck_to_structure_ratio')

    if 'MAX_SPAN_LEN_MT_048' in df.columns and 'STRUCTURE_LEN_MT_049' in df.columns:
        df_eng['span_efficiency'] = df['MAX_SPAN_LEN_MT_048'] / (df['STRUCTURE_LEN_MT_049'] + 1)
        features_created.append('span_efficiency')

    if 'STRUCTURE_LEN_MT_049' in df.columns:
        df_eng['is_long_bridge'] = (df['STRUCTURE_LEN_MT_049'] > 100).astype(int)
        df_eng['is_very_long_bridge'] = (df['STRUCTURE_LEN_MT_049'] > 300).astype(int)
        df_eng['length_category'] = pd.cut(
            df['STRUCTURE_LEN_MT_049'],
            bins=[-np.inf, 20, 60, 120, np.inf],
            labels=[0, 1, 2, 3]
        ).astype(float)
        features_created.extend(['is_long_bridge', 'is_very_long_bridge', 'length_category'])

    if 'DECK_WIDTH_MT_052' in df.columns and 'ROADWAY_WIDTH_MT_051' in df.columns:
        df_eng['deck_roadway_diff'] = df['DECK_WIDTH_MT_052'] - df['ROADWAY_WIDTH_MT_051']
        features_created.append('deck_roadway_diff')

    # ============================================================
    # 4. MAINTENANCE FEATURES
    # ============================================================
    if 'INSPECT_FREQ_MONTHS_091' in df.columns:
        df_eng['needs_frequent_inspection'] = (df['INSPECT_FREQ_MONTHS_091'] < 12).astype(int)
        df_eng['needs_very_frequent_inspection'] = (df['INSPECT_FREQ_MONTHS_091'] < 6).astype(int)
        features_created.extend(['needs_frequent_inspection', 'needs_very_frequent_inspection'])

    if 'YEAR_OF_IMP_097' in df.columns:
        df_eng['years_since_improvement'] = current_year - df['YEAR_OF_IMP_097']
        df_eng['years_since_improvement'] = df_eng['years_since_improvement'].fillna(999)
        df_eng['has_recent_improvement'] = (df_eng['years_since_improvement'] < 5).astype(int)
        df_eng['never_improved'] = (df['YEAR_OF_IMP_097'].isna()).astype(int)
        features_created.extend(['years_since_improvement', 'has_recent_improvement', 'never_improved'])

    if 'DESIGN_LOAD_031' in df.columns:
        df_eng['has_modern_design_load'] = (df['DESIGN_LOAD_031'] >= 5).astype(int)
        features_created.append('has_modern_design_load')

    # ============================================================
    # 5. INTERACTION FEATURES
    # ============================================================
    # Age × Traffic
    if 'bridge_age' in df_eng.columns and 'ADT_029' in df.columns:
        df_eng['age_traffic_interaction'] = df_eng['bridge_age'] * np.log1p(df['ADT_029'])
        features_created.append('age_traffic_interaction')

    # Age × Structure Length
    if 'bridge_age' in df_eng.columns and 'STRUCTURE_LEN_MT_049' in df.columns:
        df_eng['age_length_interaction'] = df_eng['bridge_age'] * df['STRUCTURE_LEN_MT_049']
        features_created.append('age_length_interaction')

    # Traffic × Truck
    if 'ADT_029' in df.columns and 'PERCENT_ADT_TRUCK_109' in df.columns:
        df_eng['traffic_truck_interaction'] = df['ADT_029'] * df['PERCENT_ADT_TRUCK_109']
        features_created.append('traffic_truck_interaction')

# 6. NEW FEATURES
# ============================================================

    # Maintenance burden score (age + improvement history)
    if 'bridge_age' in df_eng.columns and 'years_since_improvement' in df_eng.columns:
        df_eng['maintenance_burden'] = (
            df_eng['bridge_age'] / 100 +  # Normalize age
            df_eng['years_since_improvement'] / 100  # Normalize years
        )
        features_created.append('maintenance_burden')

    # Traffic stress score (traffic + trucks + age)
    if 'ADT_029' in df.columns and 'bridge_age' in df_eng.columns:
        traffic_norm = np.clip(df['ADT_029'] / df['ADT_029'].quantile(0.95), 0, 1)
        age_norm = np.clip(df_eng['bridge_age'] / 100, 0, 1)
        df_eng['traffic_stress_score'] = (traffic_norm + age_norm) / 2
        features_created.append('traffic_stress_score')

    # Inspection urgency (frequency + age + no improvements)
    if 'INSPECT_FREQ_MONTHS_091' in df.columns and 'bridge_age' in df_eng.columns:
        freq_urgency = 1 - (df['INSPECT_FREQ_MONTHS_091'] / 24)  # Lower freq = higher urgency
        age_urgency = np.clip(df_eng['bridge_age'] / 100, 0, 1)
        df_eng['inspection_urgency'] = np.clip((freq_urgency + age_urgency) / 2, 0, 1)
        features_created.append('inspection_urgency')

    print(f"\n{'Training' if is_train else 'Test'} features created: {len(features_created)}")
    print(f"  Age features: {len([f for f in features_created if 'age' in f or 'reconstruction' in f])}")
    print(f"  Traffic features: {len([f for f in features_created if 'traffic' in f or 'truck' in f or 'adt' in f])}")
    print(f"  Structural features: {len([f for f in features_created if 'ratio' in f or 'length' in f or 'width' in f or 'bridge' in f])}")
    print(f"  Maintenance features: {len([f for f in features_created if 'inspection' in f or 'improvement' in f or 'maintenance' in f])}")
    print(f"  Interaction features: {len([f for f in features_created if 'interaction' in f])}")
    print(f"  Score features: {len([f for f in features_created if 'score' in f or 'burden' in f or 'urgency' in f])}")

    return df_eng, features_created

# SUMMARY
    new_features = [f for f in features_created if f in df_eng.columns]

    print("\n" + "="*50)
    print(f"Feature Engineering Summary:")
    print(f"   Original features: {len(original_columns)}")
    print(f"   New features created: {len(new_features)}")
    print(f"   Total features: {len(df_eng.columns)}")

    return df_eng, new_features

# Test on small sample first
print("\nTesting feature engineering on sample...")
X_train_sample = X_train.head(1000)
X_train_eng_test, _ = create_engineered_features(X_train_sample, is_train=True)
print(f"\nTest successful! Sample shape: {X_train_eng_test.shape}")


Testing feature engineering on sample...

Training features created: 28
  Age features: 5
  Traffic features: 8
  Structural features: 6
  Maintenance features: 6
  Interaction features: 3
  Score features: 3

Test successful! Sample shape: (1000, 130)


In [5]:
"""
Apply feature engineering to full training and test sets
"""

print("APPLYING FEATURE ENGINEERING TO FULL DATASETS")

# Engineer training data
X_train_engineered, new_features_list = create_engineered_features(X_train, is_train=True)

# Engineer test data
X_test_engineered, _ = create_engineered_features(X_test, is_train=False)

print(f"\nFinal shapes:")
print(f"   X_train_engineered: {X_train_engineered.shape}")
print(f"   X_test_engineered: {X_test_engineered.shape}")

# Verify no missing values introduced
train_missing = X_train_engineered.isnull().sum().sum()
test_missing = X_test_engineered.isnull().sum().sum()

print(f"\nMissing values check:")
print(f"   Training: {train_missing}")
print(f"   Test: {test_missing}")

if train_missing > 0 or test_missing > 0:
    print("\nColumns with missing values:")
    train_missing_cols = X_train_engineered.isnull().sum()
    print(train_missing_cols[train_missing_cols > 0])

APPLYING FEATURE ENGINEERING TO FULL DATASETS

Training features created: 28
  Age features: 5
  Traffic features: 8
  Structural features: 6
  Maintenance features: 6
  Interaction features: 3
  Score features: 3

Test features created: 28
  Age features: 5
  Traffic features: 8
  Structural features: 6
  Maintenance features: 6
  Interaction features: 3
  Score features: 3

Final shapes:
   X_train_engineered: (1236125, 130)
   X_test_engineered: (624193, 130)

Missing values check:
   Training: 1
   Test: 1

Columns with missing values:
traffic_per_lane    1
dtype: int64


In [7]:
"""
Analyze engineered features
"""

print("\n" + "="*70)
print("FEATURE ANALYSIS")
print("="*70)

# Statistical summary of new features
print("\nStatistics for NEW engineered features:")
new_feature_stats = X_train_engineered[new_features_list].describe()
print(new_feature_stats)

# Check for infinite or extreme values
print("\nChecking for infinite values...")
inf_cols = []
for col in new_features_list:
    n_inf = np.isinf(X_train_engineered[col]).sum()
    if n_inf > 0:
        print(f"   {col}: {n_inf} infinite values")
        inf_cols.append(col)

if len(inf_cols) > 0:
    print(f"\nReplacing infinite values with NaN and then median...")
    for col in inf_cols:
        X_train_engineered[col].replace([np.inf, -np.inf], np.nan, inplace=True)
        X_test_engineered[col].replace([np.inf, -np.inf], np.nan, inplace=True)

        median_val = X_train_engineered[col].median()
        X_train_engineered[col].fillna(median_val, inplace=True)
        X_test_engineered[col].fillna(median_val, inplace=True)

    print(f"   Fixed {len(inf_cols)} columns")

# Correlation of new features with target
print("\nCorrelation of KEY features with target...")

# Encode target for correlation
y_train_encoded = y_train.map({'G': 0, 'F': 1, 'P': 2})

key_features = [
    'bridge_age', 'worst_condition', 'avg_condition',
    'traffic_per_lane', 'composite_risk_score',
    'age_condition_interaction', 'has_poor_component'
]

available_key_features = [f for f in key_features if f in X_train_engineered.columns]

if len(available_key_features) > 0:
    correlations = []
    for feat in available_key_features:
        corr = X_train_engineered[feat].corr(y_train_encoded)
        correlations.append({'feature': feat, 'correlation': corr})

    corr_df = pd.DataFrame(correlations).sort_values('correlation', key=abs, ascending=False)
    print(corr_df)


FEATURE ANALYSIS

Statistics for NEW engineered features:
         bridge_age  age_category  is_very_old  years_since_reconstruction  \
count  1.236125e+06     1236125.0    1236125.0                1.236125e+06   
mean   2.025028e+03           3.0          1.0                1.726368e+03   
std    7.023762e-01           0.0          0.0                7.121289e+02   
min    2.023789e+03           3.0          1.0               -7.974000e+03   
25%    2.024474e+03           3.0          1.0                2.025000e+03   
50%    2.025000e+03           3.0          1.0                2.025000e+03   
75%    2.025474e+03           3.0          1.0                2.025000e+03   
max    2.032395e+03           3.0          1.0                2.025000e+03   

       was_reconstructed       log_adt  is_high_traffic  traffic_per_lane  \
count          1236125.0  1.236125e+06     1.236125e+06      1.236124e+06   
mean                 1.0  3.549703e-01     9.990980e-02               NaN   
std    

In [9]:
"""
Visualize distributions of key engineered features by target class
"""
###Remove for now - idea changed due to data leakage: 
"""
print("VISUALIZING KEY FEATURES")

# Select top features to visualize
viz_features = [
    'bridge_age', 'worst_condition', 'avg_condition',
    'composite_risk_score', 'traffic_per_lane', 'has_poor_component'
]

available_viz_features = [f for f in viz_features if f in X_train_engineered.columns]

if len(available_viz_features) >= 4:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for idx, feat in enumerate(available_viz_features[:6]):
        # Create temporary dataframe for plotting
        plot_df = pd.DataFrame({
            'feature': X_train_engineered[feat],
            'condition': y_train
        })

        # Box plot by condition
        plot_df.boxplot(column='feature', by='condition', ax=axes[idx])
        axes[idx].set_title(f'{feat} by Bridge Condition')
        axes[idx].set_xlabel('Condition (F=Fair, G=Good, P=Poor)')
        axes[idx].set_ylabel(feat)
        plt.sca(axes[idx])
        plt.xticks(rotation=0)

    plt.tight_layout()
    plt.savefig('feature_distributions.png', dpi=300, bbox_inches='tight')
    print("\nSaved: feature_distributions.png")
    plt.show()

# Distribution of composite risk score
if 'composite_risk_score' in X_train_engineered.columns:
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    for condition in ['G', 'F', 'P']:
        mask = y_train == condition
        plt.hist(
            X_train_engineered.loc[mask, 'composite_risk_score'],
            bins=50,
            alpha=0.5,
            label=f'{condition} ({mask.sum():,})'
        )
    plt.xlabel('Composite Risk Score')
    plt.ylabel('Count')
    plt.title('Risk Score Distribution by Condition')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)

    plt.subplot(1, 2, 2)
    # Violin plot
    plot_df = pd.DataFrame({
        'risk_score': X_train_engineered['composite_risk_score'],
        'condition': y_train
    })
    sns.violinplot(data=plot_df, x='condition', y='risk_score', order=['G', 'F', 'P'])
    plt.title('Risk Score Distribution (Violin Plot)')
    plt.xlabel('Bridge Condition')
    plt.ylabel('Composite Risk Score')

    plt.tight_layout()
    plt.savefig('risk_score_analysis.png', dpi=300, bbox_inches='tight')
    print("Saved: risk_score_analysis.png")
    plt.show()
    
"""

VISUALIZING KEY FEATURES


In [21]:
"""
Save engineered datasets
"""
X_train_engineered['traffic_per_lane'].fillna(
    X_train_engineered['traffic_per_lane'].median(), 
    inplace=True
)
X_test_engineered['traffic_per_lane'].fillna(
    X_test_engineered['traffic_per_lane'].median(), 
    inplace=True
)

print("SAVING ENGINEERED DATASETS")
# Save full engineered datasets
X_train_engineered.to_csv('X_train_engineered.csv', index=False)
X_test_engineered.to_csv('X_test_engineered.csv', index=False)

print(f"\nSaved:")
print(f"   X_train_engineered.csv: {X_train_engineered.shape}")
print(f"   X_test_engineered.csv: {X_test_engineered.shape}")

# Save feature list
with open('feature_list.txt', 'w') as f:
    f.write("ORIGINAL FEATURES\n")
    f.write("="*50 + "\n")
    for feat in original_columns:
        f.write(f"{feat}\n")

    f.write("\n\nENGINEERED FEATURES\n")
    f.write("="*50 + "\n")
    for feat in new_features_list:
        f.write(f"{feat}\n")

    f.write(f"\n\nTOTAL: {len(X_train_engineered.columns)} features\n")
    f.write(f"   Original: {len(original_columns)}\n")
    f.write(f"   Engineered: {len(new_features_list)}\n")

print("Saved: feature_list.txt")

# Create engineering log
with open('engineering_log.txt', 'w') as f:
    f.write("FEATURE ENGINEERING LOG\n")
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Author: Vashishth Doshi\n\n")

    f.write(f"INPUT DATA:\n")
    f.write(f"   X_train_final.csv: {X_train.shape}\n")
    f.write(f"   X_test_final.csv: {X_test.shape}\n\n")

    f.write(f"OUTPUT DATA:\n")
    f.write(f"   X_train_engineered.csv: {X_train_engineered.shape}\n")
    f.write(f"   X_test_engineered.csv: {X_test_engineered.shape}\n\n")

    f.write(f"FEATURE CATEGORIES:\n")
    f.write(f"   1. Age Features: 5\n")
    f.write(f"   2. Condition Features: 7\n")
    f.write(f"   3. Traffic Features: 5\n")
    f.write(f"   4. Structural Features: 7\n")
    f.write(f"   5. Maintenance Features: 5\n")
    f.write(f"   6. Interaction Features: 5\n")
    f.write(f"   7. Risk Score: 2\n")
    f.write(f"\n   TOTAL NEW FEATURES: {len(new_features_list)}\n\n")

    f.write(f"KEY RATIONALE:\n")
    f.write(f"   - Bridge age is critical for deterioration prediction\n")
    f.write(f"   - Condition aggregates capture overall structural health\n")
    f.write(f"   - Traffic × Age interactions model cumulative stress\n")
    f.write(f"   - Composite risk score provides single deterioration metric\n")
    f.write(f"   - Structural ratios normalize for bridge size differences\n\n")

    f.write(f"NEXT STEPS (for teammate):\n")
    f.write(f"   1. Test these engineered features in baseline models\n")
    f.write(f"   2. Compare performance vs original features\n")
    f.write(f"   3. Identify which feature categories are most predictive\n")
    f.write(f"   4. Consider creating additional domain-specific features\n")

print("Saved: engineering_log.txt")
print(f"   - X_train_engineered.csv")
print(f"   - X_test_engineered.csv")
print(f"   - feature_list.txt")
print(f"   - engineering_log.txt")
print(f"   - feature_distributions.png")
print(f"   - risk_score_analysis.png")

SAVING ENGINEERED DATASETS

Saved:
   X_train_engineered.csv: (1236125, 130)
   X_test_engineered.csv: (624193, 130)
Saved: feature_list.txt
Saved: engineering_log.txt
   - X_train_engineered.csv
   - X_test_engineered.csv
   - feature_list.txt
   - engineering_log.txt
   - feature_distributions.png
   - risk_score_analysis.png
